# MDS Attack Detection - ML Pipeline

## Microarchitectural Data Sampling (MDS) Detection using Hardware Performance Counters

This notebook implements a complete ML pipeline for detecting MDS attacks based on the research document: **"Creating Static Data for MDS Analysis" (May 2026)**.

### Pipeline Steps:
1. **Data Generation** - Synthetic HPC dataset with benign and attack workloads
2. **Feature Engineering** - Raw, derived, MDS-specific, and temporal features
3. **Dataset Validation** - Statistical checks and separability analysis
4. **Model Training** - Multiple ML classifiers (RF, GB, LR, SVM, NN)
5. **Model Evaluation** - Metrics, confusion matrices, ROC curves
6. **Model Comparison** - Side-by-side performance comparison
7. **Multi-Class Detection** - Distinguishing MDS attack variants

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Project modules
from synthetic_data_generator import SyntheticDataGenerator
from feature_engineering import MDSFeatureEngineer
from dataset_validator import DatasetValidator
from mds_detector import MDSDetector, MultiClassMDSDetector

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("All imports successful!")

All imports successful!


---
## Step 1: Generate Synthetic Dataset

Generates 10,000 HPC samples across 8 workload classes (4 benign + 4 MDS attack variants) using Poisson distributions calibrated from published research.

In [2]:
generator = SyntheticDataGenerator(sampling_interval_ms=100, random_seed=42)
df = generator.generate_default_dataset(n_samples=10000, samples_per_run=100)

print(f"Dataset shape: {df.shape}")
print(f"Total runs: {df['run_id'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

Dataset shape: (9990, 17)
Total runs: 105

Columns: ['timestamp', 'sample_id', 'run_id', 'llc_load_misses', 'l1d_load_misses', 'branch_misses', 'branch_instr', 'instructions', 'cache_references', 'page_faults', 'context_switches', 'cpu_cycles', 'cache_miss_ratio', 'ipc', 'branch_miss_rate', 'label', 'attack_variant']

First 5 rows:


,timestamp,sample_id,run_id,llc_load_misses,l1d_load_misses,branch_misses,branch_instr,instructions,cache_references,page_faults,context_switches,cpu_cycles,cache_miss_ratio,ipc,branch_miss_rate,label,attack_variant
0,1.778754e+09,4022,40,2757,7259,585,200358,1999290,50181,151,202,3997661,0.054941,0.500115,0.002920,benign_io,
1,1.778754e+09,5412,55,6279,17955,886,398434,3996790,120106,63,58,6998915,0.052279,0.571059,0.002224,benign_mixed,
2,1.778753e+09,487,4,4759,14236,1016,499842,4996327,99924,20,54,8000587,0.047626,0.624495,0.002033,benign_cpu,
3,1.778753e+09,39,0,5197,15501,1001,500659,4999416,100452,22,68,7997315,0.051736,0.625137,0.001999,benign_cpu,
4,1.778754e+09,6795,70,60536,97316,3023,598729,4997823,100204,461,34,8997886,0.604128,0.555444,0.005049,msbds,msbds


In [3]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
class_counts = df['label'].value_counts()
colors = ['#2ecc71' if 'benign' in x else '#e74c3c' for x in class_counts.index]
class_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].set_xlabel('Class Label')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart - Binary split
attack_labels = ['msbds', 'mfbds', 'mlpds', 'mdsum']
binary_counts = pd.Series({
    'Benign': (~df['label'].isin(attack_labels)).sum(),
    'Attack': df['label'].isin(attack_labels).sum()
})
axes[1].pie(binary_counts, labels=binary_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 13})
axes[1].set_title('Binary Class Split', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('output/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Class counts:")
print(class_counts.to_string())

Class counts:
label
benign_cpu      2270
msbds           1360
mfbds           1360
benign_mem      1360
benign_io        910
benign_mixed     910
mdsum            910
mlpds            910


In [4]:
# Key HPC counter distributions: Benign vs Attack
counter_cols = ['llc_load_misses', 'l1d_load_misses', 'branch_misses',
                'page_faults', 'cache_references', 'cache_miss_ratio']

df['is_attack'] = df['label'].isin(attack_labels).map({True: 'Attack', False: 'Benign'})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(counter_cols):
    for label, color in [('Benign', '#2ecc71'), ('Attack', '#e74c3c')]:
        data = df[df['is_attack'] == label][col]
        axes[i].hist(data, bins=50, alpha=0.6, label=label, color=color, density=True)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].legend()
    axes[i].set_ylabel('Density')

plt.suptitle('HPC Counter Distributions: Benign vs Attack', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/counter_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 2: Feature Engineering

Extracting 4 tiers of features following PDF Section 4:
- **Raw Features**: Counter rates and basic ratios
- **Statistical Features**: Rolling mean, std, variance, skewness, kurtosis
- **MDS-Specific Features**: Cache-miss-to-instruction ratio, IPC anomalies, etc.
- **Temporal Features**: Autocorrelation, burst detection, trend analysis

In [5]:
engineer = MDSFeatureEngineer(window_size=10)
df_features = engineer.extract_all_features(df)

print(f"Features BEFORE engineering: {len(df.columns)}")
print(f"Features AFTER engineering:  {len(df_features.columns)}")
print(f"\nNew features added: {len(df_features.columns) - len(df.columns)}")
print(f"\nAll feature columns:")
for i, col in enumerate(df_features.columns):
    print(f"  {i+1:3d}. {col}")

Features BEFORE engineering: 18
Features AFTER engineering:  121

New features added: 103

All feature columns:
    1. timestamp
    2. sample_id
    3. run_id
    4. llc_load_misses
    5. l1d_load_misses
    6. branch_misses
    7. branch_instr
    8. instructions
    9. cache_references
   10. page_faults
   11. context_switches
   12. cpu_cycles
   13. cache_miss_ratio
   14. ipc
   15. branch_miss_rate
   16. label
   17. attack_variant
   18. is_attack
   19. llc_load_misses_rate
   20. l1d_load_misses_rate
   21. branch_misses_rate
   22. branch_instr_rate
   23. instructions_rate
   24. cache_references_rate
   25. page_faults_rate
   26. context_switches_rate
   27. llc_load_misses_rolling_mean
   28. llc_load_misses_rolling_std
   29. llc_load_misses_rolling_var
   30. llc_load_misses_rolling_min
   31. llc_load_misses_rolling_max
   32. llc_load_misses_rate_of_change
   33. llc_load_misses_cv
   34. llc_load_misses_skew
   35. llc_load_misses_kurtosis
   36. l1d_load_misses_

In [6]:
# Top features by correlation with attack label
importance = engineer.get_feature_importance_scores(df_features)
top_features = list(importance.items())[:20]

fig, ax = plt.subplots(figsize=(12, 8))
feat_names = [f[0] for f in top_features]
feat_scores = [f[1] for f in top_features]
bars = ax.barh(range(len(feat_names)), feat_scores, color='#3498db', edgecolor='black')
ax.set_yticks(range(len(feat_names)))
ax.set_yticklabels(feat_names, fontsize=10)
ax.set_xlabel('Absolute Correlation with Attack Label', fontsize=12)
ax.set_title('Top 20 Features by Correlation with Attack Label', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, score in zip(bars, feat_scores):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('output/feature_importance_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [7]:
# Correlation heatmap of top features
top_feat_names = [f[0] for f in top_features[:12]]
corr_matrix = df_features[top_feat_names].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap (Top 12 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 3: Dataset Validation

Running statistical validation (Section 9.1) and separability analysis (Section 9.2) to confirm dataset quality before training.

In [8]:
validator = DatasetValidator(df_features)
stat_results = validator.statistical_validation()

Statistical Validation Results

Class Distribution:
  benign_cpu     :   2270 (22.7%)
  msbds          :   1360 (13.6%)
  mfbds          :   1360 (13.6%)
  benign_mem     :   1360 (13.6%)
  benign_io      :    910 (9.1%)
  benign_mixed   :    910 (9.1%)
  mdsum          :    910 (9.1%)
  mlpds          :    910 (9.1%)

  Attack ratio: 0.454
  Benign ratio: 0.546

Feature Variance Check:
  Zero-variance features: ['store_to_load_fwd_sim']

Negative Counter Check:
  No negative counter values found.

NaN Values:
  llc_load_misses_rolling_std: 1 NaN values
  llc_load_misses_rolling_var: 1 NaN values
  llc_load_misses_rate_of_change: 1 NaN values
  llc_load_misses_cv: 1 NaN values
  llc_load_misses_skew: 2 NaN values
  llc_load_misses_kurtosis: 3 NaN values
  l1d_load_misses_rolling_std: 1 NaN values
  l1d_load_misses_rolling_var: 1 NaN values
  l1d_load_misses_rate_of_change: 1 NaN values
  l1d_load_misses_cv: 1 NaN values
  l1d_load_misses_skew: 2 NaN values
  l1d_load_misses_kurtosis: 3

In [9]:
sep_results = validator.separability_analysis()

Separability Analysis Results

Mann-Whitney U Tests:
  Significant features: 95 / 115

Top 10 Features by Effect Size (Cohen's d):
  cache_miss_ratio              : d = 8.9333
  page_fault_freq               : d = 8.7684
  page_faults                   : d = 8.7684
  page_faults_rate              : d = 8.7684
  branch_miss_rate              : d = 8.6941
  branch_misprediction_rate     : d = 8.6941
  l1_to_llc_miss_ratio          : d = 8.5144
  branch_misses_rate            : d = 8.2742
  branch_misses                 : d = 8.2742
  llc_load_misses_rate          : d = 7.3118

Top 10 Features by Mutual Information:
  cpu_cycles                    : MI = 2.0244
  branch_misses_rate            : MI = 1.9449
  branch_misses                 : MI = 1.9437
  ipc                           : MI = 1.8726
  branch_instr_rate             : MI = 1.8325
  branch_instr                  : MI = 1.8308
  instructions_rate             : MI = 1.7934
  instructions                  : MI = 1.7926
  cache_ref

In [10]:
# Box plots of key features by class
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
box_features = ['llc_load_misses', 'l1d_load_misses', 'branch_misses',
                'page_faults', 'cache_miss_ratio', 'branch_miss_rate']

for i, col in enumerate(box_features):
    df_features.boxplot(column=col, by='label', ax=axes[i], rot=45)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions by Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/boxplots_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4: Train All ML Models (Binary Classification)

Training 5 different classifiers and comparing their performance for binary (attack vs benign) detection. Data is split by `run_id` to prevent data leakage (PDF Section 6.3).

In [11]:
model_types = ['random_forest', 'gradient_boosting', 'logistic_regression', 'svm', 'neural_network']
results = {}
trained_detectors = {}

for mt in model_types:
    print(f"\n{'='*60}")
    print(f"  Training: {mt.upper()}")
    print(f"{'='*60}")
    
    detector = MDSDetector(model_type=mt)
    X_train, X_val, X_test, y_train, y_val, y_test = detector.split_by_run(df_features)
    detector.train(X_train, y_train, X_val, y_val)
    
    metrics = detector.evaluate(X_test, y_test)
    detector.print_evaluation_report(metrics)
    
    results[mt] = metrics
    trained_detectors[mt] = (detector, X_test, y_test)
    
print("\n\nAll models trained successfully!")


  Training: RANDOM_FOREST
Split by run_id:
  Train: 63 runs (6000 samples)
  Val: 21 runs (2070 samples)
  Test: 21 runs (1920 samples)
Training random_forest model...


Validation accuracy: 1.0000
MDS Detection Model Evaluation Report

Model Type: random_forest

Performance Metrics:
----------------------------------------------------------------------
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000
ROC AUC:   1.0000

Confusion Matrix:
----------------------------------------------------------------------
                Predicted
                Benign    Attack
Actual Benign     910       0
Actual Attack       0    1010

Detailed Classification Report:
----------------------------------------------------------------------
Class 0:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000
Class 1:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000

  Training: GRADIENT_BOOSTING
Split by run_id:
  Train: 63 runs (6100 samples)
  Val: 21 runs (1920 samples)
  Test: 21 runs (1970 samples)
Training gradient_boosting model...


Validation accuracy: 1.0000
MDS Detection Model Evaluation Report

Model Type: gradient_boosting

Performance Metrics:
----------------------------------------------------------------------
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000
ROC AUC:   1.0000

Confusion Matrix:
----------------------------------------------------------------------
                Predicted
                Benign    Attack
Actual Benign    1310       0
Actual Attack       0     660

Detailed Classification Report:
----------------------------------------------------------------------
Class 0:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000
Class 1:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000

  Training: LOGISTIC_REGRESSION
Split by run_id:
  Train: 63 runs (5920 samples)
  Val: 21 runs (1970 samples)
  Test: 21 runs (2100 samples)
Training logistic_regression model...
Validation accuracy: 1.0000
MDS Detection Model Evaluation Report

Model Type: logistic_

Validation accuracy: 1.0000
MDS Detection Model Evaluation Report

Model Type: svm

Performance Metrics:
----------------------------------------------------------------------
Accuracy:  0.9990
Precision: 0.9980
Recall:    1.0000
F1 Score:  0.9990
ROC AUC:   1.0000

Confusion Matrix:
----------------------------------------------------------------------
                Predicted
                Benign    Attack
Actual Benign    1098       2
Actual Attack       0    1000

Detailed Classification Report:
----------------------------------------------------------------------
Class 0:
  Precision: 1.0000
  Recall:    0.9982
  F1-Score:  0.9991
Class 1:
  Precision: 0.9980
  Recall:    1.0000
  F1-Score:  0.9990

  Training: NEURAL_NETWORK
Split by run_id:
  Train: 63 runs (6090 samples)
  Val: 21 runs (1970 samples)
  Test: 21 runs (1930 samples)
Training neural_network model...


Validation accuracy: 1.0000
MDS Detection Model Evaluation Report

Model Type: neural_network

Performance Metrics:
----------------------------------------------------------------------
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000
ROC AUC:   1.0000

Confusion Matrix:
----------------------------------------------------------------------
                Predicted
                Benign    Attack
Actual Benign    1070       0
Actual Attack       0     860

Detailed Classification Report:
----------------------------------------------------------------------
Class 0:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000
Class 1:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000


All models trained successfully!


---
## Step 5: Model Comparison Dashboard

Side-by-side comparison of all models across key metrics.

In [12]:
# Build comparison table
comparison_data = []
for mt in model_types:
    m = results[mt]
    comparison_data.append({
        'Model': mt.replace('_', ' ').title(),
        'Accuracy': m['accuracy'],
        'Precision': m['precision'],
        'Recall': m['recall'],
        'F1 Score': m['f1_score'],
        'ROC AUC': m.get('roc_auc', 'N/A'),
    })

comparison_df = pd.DataFrame(comparison_data)
print("Model Comparison Table:")
print(comparison_df.to_string(index=False))
comparison_df

Model Comparison Table:
              Model  Accuracy  Precision  Recall  F1 Score  ROC AUC
      Random Forest  1.000000   1.000000     1.0  1.000000      1.0
  Gradient Boosting  1.000000   1.000000     1.0  1.000000      1.0
Logistic Regression  1.000000   1.000000     1.0  1.000000      1.0
                Svm  0.999048   0.998004     1.0  0.999001      1.0
     Neural Network  1.000000   1.000000     1.0  1.000000      1.0


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Random Forest,1.000000,1.000000,1.0,1.000000,1.0
1,Gradient Boosting,1.000000,1.000000,1.0,1.000000,1.0
2,Logistic Regression,1.000000,1.000000,1.0,1.000000,1.0
3,Svm,0.999048,0.998004,1.0,0.999001,1.0
4,Neural Network,1.000000,1.000000,1.0,1.000000,1.0


In [13]:
# Bar chart comparison of metrics across models
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(comparison_df))
width = 0.2
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(14, 7))
for i, metric in enumerate(metrics_to_plot):
    values = comparison_df[metric].astype(float)
    bars = ax.bar(x + i * width, values, width, label=metric, color=colors[i], edgecolor='black')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Binary Classification: Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df['Model'], rotation=20, ha='right')
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(0.95, 1.01)

plt.tight_layout()
plt.savefig('output/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 6: Confusion Matrices for All Models

In [14]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, mt in enumerate(model_types):
    cm = np.array(results[mt]['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
    axes[i].set_title(mt.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices - Binary Classification', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('output/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 7: ROC Curves

In [15]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(10, 8))
colors_roc = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#9b59b6']

for mt, color in zip(model_types, colors_roc):
    detector, X_test, y_test = trained_detectors[mt]
    try:
        y_proba = detector.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc_score = roc_auc_score(y_test, y_proba)
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f"{mt.replace('_', ' ').title()} (AUC={auc_score:.4f})")
    except Exception as e:
        print(f"Could not plot ROC for {mt}: {e}")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig('output/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8: Feature Importance (Random Forest)

In [16]:
# Feature importance from Random Forest
rf_detector = trained_detectors['random_forest'][0]
feat_imp = rf_detector.get_feature_importance()

if feat_imp:
    sorted_imp = sorted(feat_imp.items(), key=lambda x: x[1], reverse=True)[:20]
    feat_names_rf = [f[0] for f in sorted_imp]
    feat_vals_rf = [f[1] for f in sorted_imp]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(range(len(feat_names_rf)), feat_vals_rf, color='#e67e22', edgecolor='black')
    ax.set_yticks(range(len(feat_names_rf)))
    ax.set_yticklabels(feat_names_rf, fontsize=10)
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_title('Random Forest - Top 20 Feature Importance', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    for bar, val in zip(bars, feat_vals_rf):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('output/rf_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Step 9: Multi-Class MDS Variant Detection

Training a multi-class classifier to distinguish between individual MDS attack variants (MSBDS, MFBDS, MLPDS, MDSUM) and benign workload types.

In [17]:
# Multi-class detection
mc_detector = MultiClassMDSDetector(model_type='random_forest')
X_train, X_val, X_test, y_train, y_val, y_test = mc_detector.split_by_run(df_features)
mc_detector.train(X_train, y_train, X_val, y_val)

mc_pred = mc_detector.predict(X_test)
mc_acc = accuracy_score(y_test, mc_pred)

print(f"Multi-class Accuracy: {mc_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, mc_pred))

Split by run_id:
  Train: 63 runs (6010 samples)
  Val: 21 runs (1970 samples)
  Test: 21 runs (2010 samples)
Training random_forest model...


Validation accuracy: 1.0000
Multi-class Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

  benign_cpu       1.00      1.00      1.00       500
   benign_io       1.00      1.00      1.00       310
  benign_mem       1.00      1.00      1.00       300
benign_mixed       1.00      1.00      1.00       100
       mdsum       1.00      1.00      1.00       300
       mfbds       1.00      1.00      1.00       200
       mlpds       1.00      1.00      1.00       100
       msbds       1.00      1.00      1.00       200

    accuracy                           1.00      2010
   macro avg       1.00      1.00      1.00      2010
weighted avg       1.00      1.00      1.00      2010



In [18]:
# Multi-class confusion matrix
mc_cm = confusion_matrix(y_test, mc_pred)
class_labels = sorted(y_test.unique())

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(mc_cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            xticklabels=class_labels, yticklabels=class_labels)
ax.set_title('Multi-Class Confusion Matrix (Random Forest)', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual Class', fontsize=12)
ax.set_xlabel('Predicted Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('output/multiclass_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Cross-Validation (Run-Level)

Performing 5-fold cross-validation with run-level splitting to get robust performance estimates.

In [19]:
# Cross-validation for the best model (Random Forest)
cv_detector = MDSDetector(model_type='random_forest')
X_all, y_all = cv_detector.prepare_data(df_features)
run_ids = df_features['run_id'].values

cv_results = cv_detector.cross_validate(X_all, y_all, n_folds=5, run_ids=run_ids)

print("Cross-Validation Results (Random Forest):")
print(f"  Type: {cv_results['type']}")
print(f"  Fold scores: {[f'{s:.4f}' for s in cv_results['cv_scores']]}")
print(f"  Mean accuracy: {cv_results['mean_score']:.4f} (+/- {cv_results['std_score']:.4f})")

# Visualize CV scores
fig, ax = plt.subplots(figsize=(8, 5))
folds = range(1, len(cv_results['cv_scores']) + 1)
ax.bar(folds, cv_results['cv_scores'], color='#3498db', edgecolor='black', alpha=0.8)
ax.axhline(y=cv_results['mean_score'], color='red', linestyle='--', lw=2,
           label=f"Mean: {cv_results['mean_score']:.4f}")
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('5-Fold Cross-Validation (Run-Level Split)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0.95, 1.01)

plt.tight_layout()
plt.savefig('output/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

Cross-Validation Results (Random Forest):
  Type: run_level
  Fold scores: ['1.0000', '1.0000', '1.0000', '1.0000', '1.0000']
  Mean accuracy: 1.0000 (+/- 0.0000)


---
## Summary

### Pipeline Results
- **Dataset**: ~10,000 synthetic HPC samples across 8 classes (4 benign + 4 MDS attack variants)
- **Features**: 120 engineered features from 4 tiers (raw, statistical, MDS-specific, temporal)
- **Binary Classification**: All 5 models achieved near-perfect detection (Accuracy, F1, AUC)
- **Multi-Class Detection**: Random Forest correctly identifies individual attack variants
- **Cross-Validation**: Consistent high accuracy across all folds with run-level splitting

### Key Findings
- `cache_miss_ratio`, `page_faults`, and `branch_miss_rate` are the most discriminative features
- MDS attacks produce significantly elevated LLC/L1 cache misses and page fault rates
- Run-level splitting prevents data leakage and provides honest performance estimates
- Random Forest and Gradient Boosting are the recommended models for deployment